# Chapter 10 &mdash; Derivatives as DFA States, Smart Constructors, CFG Parsing

**Concept 8 of the Chapter 10 decomposition:** *Closing Thoughts: Derivatives as DFA States, Smart Constructors, and CFG Parsing*

Derived expressions can <i>name</i> DFA states &mdash; provided smart constructors recognise a re-generated expression.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter10/Concept-Derivatives-As-DFA-States/Concept-Derivatives-As-DFA-States.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --
from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.Def_RE2NFA     import *
from jove.Def_rederiv    import *
from jove.AnimateDFA     import *
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


The closing idea. Brzozowski's theorem: an RE has only **finitely many distinct
derivatives** up to a few algebraic identities. So the derivatives **are** the states
of a DFA:

* state = an expression,
* transition on $c$ = take the derivative,
* final = nullable.

The catch is "**up to identities**". $E_c$ built naively is a *new tuple* even when it
denotes an expression already seen, so the state set never closes. **Smart
constructors** &mdash; simplifying $\emptyset+E \to E$, $\varepsilon E \to E$,
$E^{**}\to E^*$ and so on &mdash; make the re-generated expression *syntactically
identical*, and the construction terminates.

The same idea, applied to grammars, gives **parsing with derivatives** for
context-free languages (Chapter 11 and after).

## 2. Definitions

### The matcher, plus smart constructors

In [ ]:
# --- the derivative matcher, in full -------------------------------------
# AST forms produced by re2ast:
#    ('@','@')            epsilon
#    ('str', c)           a single symbol
#    ('+', (E1, E2))      union
#    ('.', (E1, E2))      concatenation
#    ('*', E)             star
#    ('!', E)             negation
#    ('&', (E1, E2))      intersection
EPS   = ('@', '@')
PHI   = ('phi', 'phi')          # the empty language -- not produced by the
                                # parser, but the derivative rules need it

def nullable(E):
    t = E[0]
    if t == '@'  : return True
    if t == 'phi': return False
    if t == 'str': return False
    if t == '+'  : return nullable(E[1][0]) or  nullable(E[1][1])
    if t == '&'  : return nullable(E[1][0]) and nullable(E[1][1])
    if t == '.'  : return nullable(E[1][0]) and nullable(E[1][1])
    if t == '*'  : return True
    if t == '!'  : return not nullable(E[1])
    raise ValueError(E)

def dv(c, E):
    t = E[0]
    if t == '@'  : return PHI
    if t == 'phi': return PHI
    if t == 'str': return EPS if E[1] == c else PHI
    if t == '+'  : return ('+', (dv(c, E[1][0]), dv(c, E[1][1])))
    if t == '&'  : return ('&', (dv(c, E[1][0]), dv(c, E[1][1])))
    if t == '*'  : return ('.', (dv(c, E[1]), E))
    if t == '!'  : return ('!', dv(c, E[1]))
    if t == '.'  :
        E1, E2 = E[1]
        left = ('.', (dv(c, E1), E2))
        return ('+', (left, dv(c, E2))) if nullable(E1) else left
    raise ValueError(E)

def matches(s, E):
    for ch in s:
        E = dv(ch, E)
    return nullable(E)

def rmatch(restr, s):
    return matches(s, re2ast(restr)[0])

# --- smart constructors: simplify as you build -----------------------------
def _parts(E, op):
    # flatten a right- or left-leaning tree of the same operator
    return _parts(E[1][0], op) + _parts(E[1][1], op) if E[0] == op else [E]

def _rebuild(parts, op, unit):
    # associativity + commutativity + idempotence: flatten, drop units,
    # de-duplicate, and rebuild in a canonical order.  Without this the
    # derivative of an intersection keeps producing NEW tuples for the
    # SAME expression, and the state set never closes.
    ps = sorted({p for p in parts if p != unit}, key=repr)
    if not ps: return unit
    E = ps[0]
    for q in ps[1:]: E = (op, (E, q))
    return E

def mkAlt(a, b):
    return _rebuild(_parts(('+', (a, b)), '+'), '+', PHI)

def mkCat(a, b):
    if a == PHI or b == PHI: return PHI
    if a == EPS: return b
    if b == EPS: return a
    return ('.', (a, b))

def mkStar(a):
    if a in (PHI, EPS): return EPS
    if a[0] == '*':     return a
    return ('*', a)

def mkAnd(a, b):
    if a == PHI or b == PHI: return PHI
    return _rebuild(_parts(('&', (a, b)), '&'), '&', None) or PHI

def mkNot(a):
    return a[1] if a[0] == '!' else ('!', a)

def sdv(c, E):
    """derivative built with smart constructors"""
    t = E[0]
    if t in ('@', 'phi'): return PHI
    if t == 'str': return EPS if E[1] == c else PHI
    if t == '+'  : return mkAlt(sdv(c, E[1][0]), sdv(c, E[1][1]))
    if t == '&'  : return mkAnd(sdv(c, E[1][0]), sdv(c, E[1][1]))
    if t == '*'  : return mkCat(sdv(c, E[1]), E)
    if t == '!'  : return mkNot(sdv(c, E[1]))
    if t == '.'  :
        E1, E2 = E[1]
        left = mkCat(sdv(c, E1), E2)
        return mkAlt(left, sdv(c, E2)) if nullable(E1) else left
    raise ValueError(E)

### Build a DFA whose states ARE expressions

In [ ]:
def deriv_dfa(restr, sigma='01', limit=200):
    E0 = re2ast(restr)[0]
    names, Q, todo, Dl = {E0: 'I' + ('F' if nullable(E0) else '')}, [E0], [E0], {}
    def name(E):
        if E not in names:
            names[E] = ('F' if nullable(E) else 'S') + str(len(names))
        return names[E]
    while todo:
        if len(Q) > limit: raise RuntimeError("did not converge")
        E = todo.pop()
        for ch in sigma:
            F = sdv(ch, E)
            if F not in names:
                name(F); Q.append(F); todo.append(F)
            Dl[(name(E), ch)] = name(F)
    F = {name(E) for E in Q if nullable(E)}
    return mk_dfa({name(E) for E in Q}, set(sigma), Dl, names[E0], F), Q

## 3. Tests

**Without** smart constructors the expression grows without bound.

In [ ]:
E = re2ast("(0+1)*01")[0]
def size(E):
    if E[0] in ('@', 'phi', 'str'): return 1
    if E[0] in ('*', '!'): return 1 + size(E[1])
    return 1 + size(E[1][0]) + size(E[1][1])
naive, smart = E, E
for i in range(1, 9):
    naive = dv('0', naive); smart = sdv('0', smart)
    print("  %d symbols: naive size %4d, smart size %2d" % (i, size(naive), size(smart)))
assert size(naive) > 5 * size(smart)

**With** them, the set of distinct derivatives is finite &mdash; and that is a DFA.

In [ ]:
D, states = deriv_dfa("(0+1)*01")
print("distinct derivatives found : %d" % len(states))
print("DFA states                 :", sorted(D["Q"]))
print("final states               :", sorted(D["F"]))
assert len(states) <= 8

The derivative DFA is isomorphic to the minimal DFA. It **is** the canonical machine.

In [ ]:
Dm = min_dfa(nfa2dfa(re2nfa("(0+1)*01")))
print("derivative DFA : %d states" % len(min_dfa(D)["Q"]))
print("min_dfa route  : %d states" % len(Dm["Q"]))
assert iso_dfa(min_dfa(D), Dm)
print("isomorphic? ", iso_dfa(min_dfa(D), Dm))

It works across a batch of expressions.

In [ ]:
for r in ["0*1", "(01)*", "0*1*", "(0+1)*1(0+1)(0+1)", "(0+1)*11"]:
    D, st = deriv_dfa(r)
    Dm = min_dfa(nfa2dfa(re2nfa(r)))
    print("%-22s derivatives %2d, minimal %2d, iso %s"
          % (r, len(st), len(Dm["Q"]), iso_dfa(min_dfa(D), Dm)))
    assert iso_dfa(min_dfa(D), Dm)

And it handles `!` and `&`, which `re2nfa` cannot parse at all.

In [ ]:
D, st = deriv_dfa("!((0+1)*01)")
print("!( (0+1)*01 ) : %d derivative states" % len(st))
Dm = comp_dfa(min_dfa(nfa2dfa(re2nfa("(0+1)*01"))))
assert langeq_dfa(min_dfa(D), min_dfa(Dm))
print("same language as comp_dfa's answer :", langeq_dfa(min_dfa(D), min_dfa(Dm)))
print()
D2, st2 = deriv_dfa("((0+1)*0(0+1)*)&(!((0+1)*11(0+1)*))")
print("an intersection with a negation : %d derivative states" % len(st2))
from itertools import product
spec = lambda s: ('0' in s) and ('11' not in s)
strs = [''.join(p) for k in range(9) for p in product('01', repeat=k)]
assert all(accepts_dfa(D2, s) == spec(s) for s in strs)
print("and it recognises exactly the intended language.")

The forward pointer: the same trick parses **grammars**.

In [ ]:
print("derivative of an RE      -> what is left to match  (this chapter)")
print("derivative of a GRAMMAR  -> what is left to parse   (Chapter 11 onward)")
print()
print("Same shape, same termination worry, same fix: smart constructors.")

## 4. Animation

A DFA whose states were never enumerated &mdash; they are expressions.

*(The `display(HTML(...))` line loads the toolbar's font-awesome icons. Keep it last in the cell &mdash; it must be there for the controls to appear.)*

In [ ]:
from jove.AnimateDFA import *
AnimateDFA(min_dfa(deriv_dfa('(0+1)*01')[0]), FuseEdges=True)
display(HTML('<link rel="stylesheet" href="//stackpath.bootstrapcdn.com/font-awesome/4.7.0/css/font-awesome.min.css"/>'))

## 5. Exercises


1. Which identity is doing the most work? Remove one and see if `deriv_dfa` still converges.
2. Prove that $E^{**} = E^*$ from the definition of star.
3. Read up on "parsing with derivatives". What plays the role of `nullable` there?

In [ ]:
# Your work for the exercises above.